# 02 · 递进式回归建模复现

**复现论文表4.4**：以播放量对数 `log_view` 为因变量，递进加入控制变量 → 标题特征 → 封面特征 → 双模态特征，
OLS 使用 **按 uid 聚类的稳健标准误**（同一 UP 主的多条视频误差相关）。

本 Notebook 公式与论文附录 C 完全一致（含发布月份/星期、领域与层级虚拟变量）。

In [1]:
import sys
from pathlib import Path
if (Path.cwd() / 'src').exists():
    sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from src import paths

df = pd.read_csv(paths.MODELING_CSV, encoding='utf-8-sig', low_memory=False)
print('样本量:', len(df))

BASE = '''log_view ~ log_fans + log_age + log_duration + tag_count + description_len
+ is_collection + copyright + C(pub_month) + C(pub_weekday) + C(domain) + C(level)'''
TITLE = ['title_char_len','title_question_type','title_exclamation_type',
         'title_has_number','title_sentiment_intensity',
         'curiosity_marker_count','practical_value_word_count']
COVER = ['image_contrast','image_brightness','image_colorfulness','visual_complexity',
         'cover_has_text','cover_text_area_ratio','face_presence']

def fit(formula, label):
    m = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['uid']})
    print(f'{label}: N={int(m.nobs)}, R2={m.rsquared:.4f}, 调整R2={m.rsquared_adj:.4f}')
    return m

m0 = fit(BASE, 'Model0 仅控制变量')
m1 = fit(BASE + ' + ' + ' + '.join(TITLE), 'Model1 +标题特征')
m2 = fit(BASE + ' + ' + ' + '.join(TITLE + COVER), 'Model2 +封面特征')

样本量: 4129
Model0 仅控制变量: N=4129, R2=0.6251, 调整R2=0.6224
Model1 +标题特征: N=4129, R2=0.6271, 调整R2=0.6238
Model2 +封面特征: N=4129, R2=0.6307, 调整R2=0.6268


In [2]:
# 双模态变量先标准化（论文：连续核心解释变量进入模型前标准化/中心化）
for v in ['image_title_fit','title_cover_text_overlap']:
    df[v+'_z'] = (df[v] - df[v].mean()) / df[v].std(ddof=0)
df['inter_z'] = df['image_title_fit_z'] * df['title_cover_text_overlap_z']
BIMODAL = ['image_title_fit_z + I(image_title_fit_z**2)',
           'title_cover_text_overlap_z', 'sentiment_consistency',
           'sentiment_conflict', 'inter_z']
m3 = fit(BASE + ' + ' + ' + '.join(TITLE + COVER + BIMODAL), 'Model3/4 +双模态及交互')

Model3/4 +双模态及交互: N=4129, R2=0.6353, 调整R2=0.6309


In [3]:
# 核心变量系数汇总（与论文表4.4数字一致：R2=0.6307，数字系数0.1904*）
def star(p):
    return '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.1 else ''))
rows = []
for name, m in [('Model1', m1), ('Model2', m2), ('Model3/4', m3)]:
    for v in TITLE + COVER + ['title_cover_text_overlap_z','inter_z','image_title_fit_z']:
        key = v if v in m.params.index else None
        if key is not None:
            rows.append({'模型':name,'变量':v,'系数':round(m.params[key],4),
                         '聚类稳健SE':round(m.bse[key],4),'p值':round(m.pvalues[key],3),
                         '显著性':star(m.pvalues[key])})
res = pd.DataFrame(rows)
display(res.pivot_table(index='变量', columns='模型', values='系数', aggfunc='first')
          .reindex(TITLE+COVER+['title_cover_text_overlap_z','image_title_fit_z','inter_z']).dropna(how='all'))
print('\nModel3/4 关键双模态变量：')
sub = res[(res['模型']=='Model3/4') & res['变量'].isin(['title_cover_text_overlap_z','inter_z','title_has_number'])]
display(sub.reset_index(drop=True))

模型,Model1,Model2,Model3/4
变量,,,
title_char_len,-0.0013,-0.0010,-0.0039
title_question_type,0.0082,-0.0407,-0.0578
title_exclamation_type,-0.1788,-0.2106,-0.2145
title_has_number,0.1917,0.1904,0.1838
title_sentiment_intensity,-0.0385,-0.0635,-0.1841
curiosity_marker_count,0.0295,0.0278,0.0341
practical_value_word_count,-0.0403,-0.0599,-0.0744
image_contrast,NaN,0.0080,0.0072
image_brightness,NaN,0.0006,0.0004



Model3/4 关键双模态变量：


,模型,变量,系数,聚类稳健SE,p值,显著性
0,Model3/4,title_has_number,0.1838,0.1092,0.092,*
1,Model3/4,title_cover_text_overlap_z,-0.0635,0.1049,0.545,
2,Model3/4,inter_z,-0.1571,0.0520,0.002,***


## 建模结论

1. **UP 主粉丝量对数**系数约 1.09，各模型均 p<0.001——账号基础盘是传播第一要素；
2. **标题含数字**系数 0.190（Model2，p<0.10），方向在各模型稳定为正，
   说明"结构清晰、收益明确"的数字清单式标题有助于用户快速判断内容；
3. 疑问式标题、情绪强度、实用价值词与单一封面视觉特征（亮度/色彩/复杂度/人脸）**均不显著**；
4. **图文语义匹配度 × 文字重叠度交互项显著为负（p<0.01）**：当标题与封面在语义和文字层面
   同时高度重复时传播效果下降——有效入口设计是"主题一致、信息互补"，而非图文完全一致；
5. 模型解释力随变量组递进提升，R² 从 0.6251 升至 0.6353，稳健性检验（见 step10）方向一致。

> 完整 20+ 变量结果、异质性分组与稳健性检验见 `outputs/regression_results.xlsx`
> 与 `reports/analysis_reports/` 下的文字报告。
>
> 口径说明：本 Notebook 复现论文正文表 4.4（R²≈0.635）；完整管道 step08 的更全规格
> （含标准化变量）在 xlsx 中 R²≈0.645→0.655，两种口径结论一致。